# Támadási stílusok a 2019-es és 2023-as női világbajnokságon
StatsBomb Open Data → labdabirtoklások → 6 változókészlet (S1–S6) → K-means → profilozás → csapatszintű próbák (H1–H6)

In [ ]:
%pip install -q statsbombpy factor_analyzer

In [ ]:
import ast
import numpy as np
import pandas as pd
from statsbombpy import sb
from scipy.stats import spearmanr, wilcoxon, friedmanchisquare
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import PowerTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 200)

## Adatok: StatsBomb labdabirtoklások

In [ ]:
SEASONS = {30: 2019, 107: 2023}                     # WWC (competition_id = 72)
ATTACK_TYPES = {'Pass', 'Carry', 'Dribble', 'Shot'}

games = pd.concat([sb.matches(competition_id=72, season_id=s).assign(year=y) for s, y in SEASONS.items()])
year_of = games.set_index('match_id')['year']

def get_possessions(match_id):
    ev = sb.events(match_id=match_id)
    ev = ev[ev['period'] <= 4]                                                  # büntetőpárbaj nélkül
    a = ev[ev['type'].isin(ATTACK_TYPES) & (ev['team'] == ev['possession_team'])]
    a = a[a.groupby('possession')['id'].transform('size') >= 3].sort_values('index').copy()
    a['sequence_id'] = f'{match_id}_' + a['possession'].astype(str)
    a['year'] = year_of[match_id]
    return a

COLS = ['sequence_id', 'year', 'possession_team', 'index', 'id', 'play_pattern', 'type', 'duration',
        'location', 'pass_end_location', 'carry_end_location', 'pass_length', 'pass_angle',
        'pass_outcome', 'pass_cross', 'pass_through_ball']
df = (pd.concat([get_possessions(m) for m in games['match_id'].unique()])[COLS]
      .drop_duplicates('id').sort_values(['sequence_id', 'index']).reset_index(drop=True))
df[['pass_cross', 'pass_through_ball']] = df[['pass_cross', 'pass_through_ball']].fillna(False).astype(bool)

def split_xy(s):
    xy = [v[:2] if isinstance(v, (list, tuple, np.ndarray)) and len(v) >= 2 else (np.nan, np.nan)
          for v in s.map(lambda v: ast.literal_eval(v) if isinstance(v, str) else v)]
    xy = np.array(xy, dtype=float)
    return xy[:, 0], xy[:, 1]

for p, c in [('', 'location'), ('pass_end_', 'pass_end_location'), ('carry_end_', 'carry_end_location')]:
    df[f'{p}x'], df[f'{p}y'] = split_xy(df[c])

def zone(v, bins, labels):
    out = np.select([v < bins[0], v < bins[1]], labels[:2], default=labels[2]).astype(object)
    out[np.isnan(v)] = np.nan
    return out

df['x_zone'] = zone(df['x'].to_numpy(), (40, 80), ['defensive', 'middle', 'attacking'])
df['y_zone'] = zone(df['y'].to_numpy(), (26.67, np.nextafter(53.33, np.inf)), ['left', 'center', 'right'])
print(df['sequence_id'].nunique(), 'birtoklás,', len(df), 'esemény')

/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: 

16463 birtoklás, 177055 esemény


## Birtoklásszintű változók

In [ ]:
LONG_PASS_YD = 35                               # 35 yard
SET_PIECES = ['From Corner', 'From Free Kick', 'From Throw In']

def ball_path(g):
    pts = []
    for e in g.itertuples():
        pts.append((e.x, e.y))
        if pd.notna(e.pass_end_x):
            pts.append((e.pass_end_x, e.pass_end_y))
        elif pd.notna(e.carry_end_x):
            pts.append((e.carry_end_x, e.carry_end_y))
    return np.array(pts, dtype=float)

def get_features(g):
    dur = g['duration'].sum()
    p = g[g['type'] == 'Pass']
    n = len(p)
    sh = lambda m: m.sum() / n if n else 0.0
    sx, ex, ang = p['x'], p['pass_end_x'], p['pass_angle'].abs()
    path = ball_path(g)
    length = np.linalg.norm(np.diff(path, axis=0), axis=1).sum()
    progress = g['x'].iloc[-1] - g['x'].iloc[0]
    ts = lambda m: g.loc[m, 'duration'].sum() / dur if dur > 0 else np.nan
    d3, m3 = ts(g['x_zone'] == 'defensive'), ts(g['x_zone'] == 'middle')
    side_ev = g['y_zone'].isin(['left', 'right'])
    side = ts(side_ev)
    return {
        'year': g['year'].iloc[0], 'team': g['possession_team'].iloc[0], 'play_pattern': g['play_pattern'].iloc[0],
        'shot': int((g['type'] == 'Shot').any()),
        'total_duration': dur, 'total_passes': n, 'progress': progress, 'length': length,
        'direct_speed': progress / dur if dur > 0 else np.nan, 'speed': length / dur if dur > 0 else np.nan,
        'tempo': n / dur if dur > 0 else np.nan,
        'width': np.nanmax(path[:, 1]) - np.nanmin(path[:, 1]), 'absolute_width': np.nanmax(np.abs(path[:, 1] - 40)),
        'd3_time_share': d3, 'm3_time_share': m3, 'a3_time_share': 1 - (d3 + m3),
        'side_time_share': side, 'central_time_share': 1 - side,
        'center_events': (~side_ev).mean(), 'side_events': side_ev.mean(),        # események aránya
        'pass_d3_share': sh(sx < 40), 'pass_a3_share': sh(sx >= 80),
        'fwd_pass_share': sh(ang < np.pi / 4), 'back_pass_share': sh(ang > 3 * np.pi / 4),
        'pass_d3_m3_share': sh((sx < 40) & (ex >= 40) & (ex < 80)), 'pass_d3_a3_share': sh((sx < 40) & (ex >= 80)),
        'long_pass_share': sh(p['pass_length'] > LONG_PASS_YD),
        'pass_success_share': sh(p['pass_outcome'].isna()),
        'pass_directness': float(np.degrees(ang.mean())) if n else 0.0,
        'dribble_count': int((g['type'] == 'Dribble').sum()),
        'through_ball_count': int(g['pass_through_ball'].sum()), 'cross_count': int(g['pass_cross'].sum()),
    }

att = pd.DataFrame([get_features(g) for _, g in df.groupby('sequence_id')])
att = att[~(att['play_pattern'].isin(SET_PIECES) & (att['total_duration'] < 15))]
att = att.dropna(subset=['tempo', 'direct_speed']).reset_index(drop=True)
att['team'] = att['team'].str.replace(r" (Women's|W)$", '', regex=True).str.strip()
print(len(att), 'labdabirtoklás')

12245 labdabirtoklás


## FIFA-világranglista (2019.03.29., 2023.06.09.)

In [ ]:
RANKING = {
    2019: {'United States': 1, 'Germany': 2, 'England': 3, 'France': 4, 'Canada': 5, 'Australia': 6,
           'Japan': 7, 'Netherlands': 8, 'Sweden': 9, 'Brazil': 10, 'Norway': 12, 'Spain': 13,
           'Korea Republic': 14, 'Italy': 15, 'China PR': 16, 'New Zealand': 19, 'Scotland': 20,
           'Thailand': 34, 'Argentina': 37, 'Nigeria': 38, 'Chile': 39, 'Cameroon': 46,
           'South Africa': 49, 'Jamaica': 53},
    2023: {'United States': 1, 'Germany': 2, 'Sweden': 3, 'England': 4, 'France': 5, 'Spain': 6,
           'Canada': 7, 'Brazil': 8, 'Netherlands': 9, 'Australia': 10, 'Japan': 11, 'Norway': 12,
           'Denmark': 13, 'China PR': 14, 'Italy': 16, 'Korea Republic': 17, 'Switzerland': 20,
           'Portugal': 21, 'Republic of Ireland': 22, 'Colombia': 25, 'New Zealand': 26, 'Argentina': 28,
           'Vietnam': 32, 'Costa Rica': 36, 'Nigeria': 40, 'Jamaica': 43, 'Philippines': 46,
           'Panama': 52, 'Haiti': 53, 'South Africa': 54, 'Morocco': 72, 'Zambia': 77},
}
team_rank = pd.DataFrame([(y, t, r) for y, d in RANKING.items() for t, r in d.items()], columns=['year', 'team', 'rank'])
rank_change = team_rank.pivot(index='team', columns='year', values='rank').dropna()
rank_change['rank_diff'] = rank_change[2019] - rank_change[2023]          # > 0: előrelépett

## Változókészletek, stíluskódok, paraméterek

In [ ]:
SPECS = {
    'S1 Plakias': ['total_passes', 'total_duration', 'long_pass_share', 'pass_success_share', 'tempo',
                   'dribble_count', 'cross_count', 'side_time_share', 'm3_time_share', 'a3_time_share'],
    'S2 Peng': ['cross_count', 'through_ball_count', 'total_passes', 'pass_d3_share', 'pass_a3_share',
                'pass_success_share'],
    'S3 Fernandez-Navarro': ['total_duration', 'm3_time_share', 'a3_time_share', 'side_time_share', 'fwd_pass_share',
                             'back_pass_share', 'pass_d3_m3_share', 'pass_d3_a3_share', 'cross_count'],
    'S4 Opta': ['total_passes', 'progress', 'length', 'speed', 'direct_speed', 'width', 'absolute_width'],
    'S5 Football Manager 2024': ['width', 'through_ball_count', 'center_events', 'side_events', 'm3_time_share',
                                 'a3_time_share', 'pass_directness', 'tempo', 'cross_count', 'dribble_count'],
    'S6 all': ['total_passes', 'total_duration', 'long_pass_share', 'pass_success_share', 'tempo', 'dribble_count',
               'cross_count', 'side_time_share', 'm3_time_share', 'a3_time_share', 'through_ball_count',
               'pass_d3_share', 'pass_a3_share', 'fwd_pass_share', 'back_pass_share', 'pass_d3_m3_share',
               'pass_d3_a3_share', 'progress', 'length', 'speed', 'direct_speed', 'width', 'absolute_width',
               'center_events', 'side_events', 'pass_directness'],
}
STYLE = {                                               # K-means címke -> stíluskód
    'S1 Plakias':               {2: 'LT', 4: 'B', 1: 'R', 3: 'Rvk', 0: 'Dvk', 5: 'E'},
    'S2 Peng':                  {0: 'Vvk', 1: 'LTkt', 2: 'MK'},
    'S3 Fernandez-Navarro':     {1: 'V', 2: 'B', 3: 'Dvk', 0: 'D', 4: 'HI'},
    'S4 Opta':                  {1: 'LT', 0: 'D', 2: 'R'},
    'S5 Football Manager 2024': {1: 'E', 4: 'B', 3: 'Rvk', 0: 'LT', 2: 'Dvk', 5: 'MK'},
    'S6 all':                   {2: 'LT', 0: 'Dvk', 1: 'Rkt'},
}
K_RANGE, B_SEL, N_PERM = range(3, 7), 20, 20
THRESH = {'ARI átlag': 0.80, 'ARI min': 0.65, 'legkisebb Jaccard': 0.75}
THR = 0.5                                               # centroid |z| küszöb (Cohen, 1988)
RARE = ['dribble_count', 'through_ball_count', 'cross_count']
ATTR = ['width', 'absolute_width', 'side_time_share', 'central_time_share', 'tempo', 'speed', 'direct_speed',
        'pass_success_share', 'total_duration', 'total_passes', 'length', 'progress']

## Klaszterezés és profilozás

In [ ]:
yj = lambda X: PowerTransformer(method='yeo-johnson').fit_transform(X)

def prepare(feats):
    X = att[feats].loc[:, att[feats].std() > 0]
    X = pd.DataFrame(yj(X), columns=X.columns)
    dropped = []
    while X.shape[1] > 1:                                                   # |r| >= 0,90: a nagyobb átlagos |r|-ű ki
        r = X.corr().abs().where(~np.eye(X.shape[1], dtype=bool))
        if r.max().max() < 0.90:
            break
        a, b = r.stack().idxmax()
        d = a if r[a].mean() >= r[b].mean() else b
        dropped.append(d)
        X = X.drop(columns=d)
    kmo, bart_p = calculate_kmo(X)[1], calculate_bartlett_sphericity(X)[1]
    n_pc = None
    if kmo >= 0.5 and bart_p < 0.05:
        pf = PCA().fit(X)
        n_pc = max(int((pf.explained_variance_ > 1).sum()),
                   int(np.argmax(np.cumsum(pf.explained_variance_ratio_) >= 0.70) + 1))
    return X, dropped, kmo, bart_p, n_pc

embed = lambda A, n_pc: PCA(n_components=n_pc, random_state=42).fit_transform(A) if n_pc else np.asarray(A)

def select_k(X, n_pc):
    Z = embed(X, n_pc)
    n, rows, labels = len(Z), [], {}
    for k in K_RANGE:
        ref = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(Z)
        labels[k] = ref
        rb, aris, jacc = np.random.default_rng(42), [], np.full((B_SEL, k), np.nan)
        for b in range(B_SEL):
            idx = np.unique(rb.choice(n, n, replace=True))
            km = KMeans(n_clusters=k, random_state=b, n_init=10).fit(Z[idx])
            aris.append(adjusted_rand_score(ref, km.predict(Z)))
            lb, lr = km.labels_, ref[idx]
            for c in range(k):
                jacc[b, c] = max(((lr == c) & (lb == j)).sum() / ((lr == c) | (lb == j)).sum() for j in range(k))
        rows.append({'k': k, 'sziluett': silhouette_score(Z, ref), 'ARI átlag': np.mean(aris),
                     'ARI min': np.min(aris), 'legkisebb Jaccard': np.nanmean(jacc, axis=0).min()})
    t = pd.DataFrame(rows).set_index('k')
    t['megfelel'] = np.all([t[c] >= v for c, v in THRESH.items()], axis=0)
    cand = t[t['megfelel']]
    k_sel = int(cand['sziluett'].idxmax()) if len(cand) else None
    return t, k_sel, (labels[k_sel] if k_sel else None)

def perm_test(X, n_pc, k, sil_obs):
    rng, null = np.random.default_rng(123), []
    for _ in range(N_PERM):
        Zp = embed(np.column_stack([rng.permutation(c) for c in X.to_numpy().T]), n_pc)
        null.append(silhouette_score(Zp, KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(Zp)))
    null = np.array(null)
    return (sil_obs - null.mean()) / null.std(), ((null >= sil_obs).sum() + 1) / (N_PERM + 1)

def extra_vars(active, dropped):                        # †: kiesett (|r| >= 0,90) és származtatott referenciák
    have, extra = set(active) | set(dropped), list(dropped)
    if {'m3_time_share', 'a3_time_share'} <= have:
        extra.append('d3_time_share')
    if 'side_time_share' in have:
        extra.append('central_time_share')
    return [v for v in dict.fromkeys(extra) if v not in active]

zmeans = lambda cols, lab: pd.DataFrame(yj(att[cols]), columns=cols).groupby(lab).mean()   # centroid z

def profile(lab, active, extra):
    cols = [c for c in active + extra if att[c].std() > 0]
    z = zmeans(cols, lab)
    med, med_all = att[cols].groupby(lab).median(), att[cols].median()
    occ, occ_all = (att[cols] > 0).groupby(lab).mean() * 100, (att[cols] > 0).mean() * 100
    def fmt(c, v):
        name = v + ('†' if v in extra else '')
        if v in RARE:
            return f'{name} {occ.loc[c, v]:.0f}% [{occ_all[v]:.0f}%]'
        return f'{name} {med.loc[c, v]:.2f} [{med_all[v]:.2f}]'
    desc = {c: ', '.join(fmt(c, v) for v, a in z.loc[c].abs().sort_values(ascending=False).items() if a >= THR) or '–'
            for c in z.index}
    return pd.DataFrame({'n': pd.Series(lab).value_counts(), 'jellemző változók: medián [teljes minta]': pd.Series(desc)})

def style_matrix(lab, active, extra):
    use = active + extra
    first = lambda vs: next((v for v in vs if v in use), None)
    v3, t3 = first(['d3_time_share', 'pass_d3_share']), first(['a3_time_share', 'pass_a3_share'])
    side = first(['side_time_share', 'side_events'])
    attr = [a for a in ATTR if a in use]
    z = zmeans(list(dict.fromkeys(v for v in [v3, t3, side, *attr] if v)), lab)
    rows = {}
    for c in z.index:
        zv, zt = (z.loc[c, v] if v else np.nan for v in (v3, t3))
        zs = z.loc[c, side] if side else np.nan
        rows[c] = {'z V3': zv, 'z T3': zt,
                   'terület': 'V3+K3' if zv >= THR or zt <= -THR else 'K3+T3' if zt >= THR or zv <= -THR else 'nem meghatározó',
                   'z sáv': zs, 'sáv': 'szél' if zs >= THR else 'tengely' if zs <= -THR else '–',
                   'jelzők': ', '.join(f'{a} {z.loc[c, a]:+.2f}' for a in attr if abs(z.loc[c, a]) >= THR) or '–'}
    return pd.DataFrame(rows).T

## Csapatszintű próbák (H2–H6)

In [ ]:
holm = lambda p: multipletests(p, method='holm')[1]

def wilcox(d):
    try:
        return wilcoxon(d)
    except ValueError:                                  # minden különbség 0
        return np.nan, 1.0

def paired_change(tab):                                 # 2023 - 2019, százalékpont, párosított Wilcoxon, Holm
    rows = {}
    for c in sorted(tab.columns.get_level_values(0).unique()):
        d = ((tab[(c, 2023)] - tab[(c, 2019)]) * 100).dropna()
        W, p = wilcox(d)
        rows[c] = {'n csapat': len(d), 'medián Δ (p.p.)': d.median(), 'W': W, 'p': p}
    r = pd.DataFrame.from_dict(rows, orient='index')
    r['p (Holm)'] = holm(r['p'])
    return r.drop(columns='p')

def team_tests(lab):
    a = att.assign(cl=lab)
    cls = sorted(a['cl'].unique())
    shares = pd.crosstab([a['team'], a['year']], a['cl'], normalize='index')
    rate = a.pivot_table(index=['team', 'year'], columns='cl', values='shot', aggfunc='mean')
    h3, h3_ph = {}, {}
    for y in sorted(a['year'].unique()):
        ry = rate.xs(y, level='year').dropna()
        chi2, p = friedmanchisquare(*[ry[c] for c in cls])
        h3[y] = {'n csapat': len(ry), 'Friedman chi2': chi2, 'p': p, 'Kendall W': chi2 / (len(ry) * (len(cls) - 1)),
                 'medián lövésarány %': ', '.join(f'{c}: {v:.1f}' for c, v in (ry.median() * 100).sort_values(ascending=False).items())}
        ph = {}
        for i, ci in enumerate(cls):
            for cj in cls[i + 1:]:
                d = (ry[ci] - ry[cj]) * 100
                W, pp = wilcox(d)
                ph[f'{ci}–{cj}'] = {'medián Δ (p.p.)': d.median(), 'W': W, 'p': pp}
        ph = pd.DataFrame.from_dict(ph, orient='index')
        ph['p (Holm)'] = holm(ph['p'])
        h3_ph[y] = ph.drop(columns='p')
    h5 = {}
    ty = shares.reset_index().merge(team_rank, on=['team', 'year'])
    for y, d in ty.groupby('year'):
        sp = [spearmanr(d[c], d['rank']) for c in cls]
        h5[(y, 'n')], h5[(y, 'rho')] = len(d), [s[0] for s in sp]
        h5[(y, 'p (Holm)')] = holm([s[1] for s in sp])
    w = shares.unstack('year')
    tvd = (0.5 * (w.xs(2023, axis=1, level='year') - w.xs(2019, axis=1, level='year')).abs()).dropna().sum(axis=1)
    ch = rank_change[['rank_diff']].join(tvd.rename('tvd'), how='inner').dropna()
    rho, p6 = spearmanr(ch['rank_diff'], ch['tvd'])
    return {'H2': paired_change(shares.unstack('year')),
            'H3': pd.DataFrame.from_dict(h3, orient='index'), 'H3 post hoc': pd.concat(h3_ph, names=['év']),
            'H4': paired_change(a.pivot_table(index='team', columns=['cl', 'year'], values='shot', aggfunc='mean')),
            'H5': pd.DataFrame(h5, index=cls),
            'H6': {'n csapat': len(ch), 'medián TVD': ch['tvd'].median(), 'rho': rho, 'p': p6}}

## Futtatás mind a 6 változókészletre

In [ ]:
R = {}
for s, feats in SPECS.items():
    X, dropped, kmo, bart_p, n_pc = prepare(feats)
    t, k, lab = select_k(X, n_pc)
    r = R[s] = {'változók': list(X.columns), 'kiesett': dropped, 'sel': t, 'k': k}
    if k:
        r['perm z'], r['perm p'] = perm_test(X, n_pc, k, t.loc[k, 'sziluett'])
        if r['perm p'] >= 0.05:
            r['k'] = None
    print('=' * 110 + f'\n{s} | változók: {", ".join(X.columns)}' + (f' | kiesett: {", ".join(dropped)}' if dropped else ''))
    print(f'KMO = {kmo:.3f}, Bartlett p = {bart_p:.3g}, PC-k: {n_pc or "nincs PCA"}')
    print(t.round(3).to_string())
    if not r['k']:
        print('H1: nem klaszterezhető')
        continue
    print(f'H1: k = {k}, permutációs z = {r["perm z"]:.1f}, p = {r["perm p"]:.3f}')
    lab = np.array([STYLE[s][c] for c in lab])
    extra = extra_vars(r['változók'], dropped)
    r['cls'] = lab
    r['profil'] = profile(lab, r['változók'], extra)
    r['mátrix'] = style_matrix(lab, r['változók'], extra)
    r.update(team_tests(lab))
    print('\nProfil (centroid |z| >= 0,5; †: kiesett / származtatott)\n' + r['profil'].to_string())
    print('\nStílusmátrix\n' + r['mátrix'].to_string(float_format='{:+.2f}'.format))
    print('\nH2 – stílusarány változása\n' + r['H2'].round(3).to_string())
    print('\nH3 – lövésarány a stílusok között (Friedman)\n' + r['H3'].to_string())
    print('\nH3 post hoc (Wilcoxon, Holm)\n' + r['H3 post hoc'].round(3).to_string())
    print('\nH4 – lövésarány változása\n' + r['H4'].round(3).to_string())
    print('\nH5 – stílusarány ~ FIFA-helyezés (Spearman)\n' + r['H5'].round(3).to_string())
    print('\nH6 – TVD ~ helyezésváltozás: ' + ', '.join(f'{k} {v:.3f}' if isinstance(v, float) else f'{k} {v}' for k, v in r['H6'].items()))

S1 Plakias | változók: total_passes, total_duration, long_pass_share, pass_success_share, tempo, dribble_count, cross_count, side_time_share, m3_time_share, a3_time_share
KMO = 0.362, Bartlett p = 0, PC-k: nincs PCA
   sziluett  ARI átlag  ARI min  legkisebb Jaccard  megfelel
k                                                           
3     0.161      0.960    0.925              0.973      True
4     0.173      0.979    0.963              0.982      True
5     0.178      0.978    0.962              0.976      True
6     0.179      0.967    0.941              0.913      True
H1: k = 6, permutációs z = 22.1, p = 0.048

Profil (centroid |z| >= 0,5; †: kiesett / származtatott)
        n                                                                                                                                                                    jellemző változók: medián [teljes minta]
B    1677                                                                       cross_count 100% [14%],

## Összefoglalás

In [ ]:
fp = lambda p: '< 0,001' if p < 0.001 else f'{p:.3f}'

def sig(tab):
    s = tab[tab['p (Holm)'] < 0.05]
    return '; '.join(f"{i} {x['medián Δ (p.p.)']:+.1f} p.p. (p {fp(x['p (Holm)'])})" for i, x in s.iterrows()) or 'nincs'

def hyp_row(r):
    if not r['k']:
        return {'H1': 'nem klaszterezhető'}
    t, h5 = r['sel'].loc[r['k']], r['H5']
    return {'H1': f"k={r['k']}, sziluett {t['sziluett']:.2f}, ARI {t['ARI átlag']:.2f} (min {t['ARI min']:.2f}), "
                  f"Jaccard {t['legkisebb Jaccard']:.2f}, perm. p {fp(r['perm p'])}",
            'H2': sig(r['H2']),
            'H3': '; '.join(f"{y}: χ² {x['Friedman chi2']:.1f}, p {fp(x['p'])}, W {x['Kendall W']:.2f}" for y, x in r['H3'].iterrows()),
            'H4': sig(r['H4']),
            'H5': '; '.join(f'{y}: ' + (', '.join(f"{c} {h5.loc[c, (y, 'rho')]:+.2f}" for c in h5.index
                                                  if h5.loc[c, (y, 'p (Holm)')] < 0.05) or 'nincs') for y in (2019, 2023)),
            'H6': f"ρ {r['H6']['rho']:+.2f}, p {fp(r['H6']['p'])}, n {r['H6']['n csapat']}"}

display(pd.DataFrame({s: hyp_row(r) for s, r in R.items()}).T.fillna('–'))

# külső ellenőrzés: játékhelyzet klaszterenként (%); a bennmaradt (>= 15 s) pontrúgások nyílt játéknak számítanak
pp = att['play_pattern'].replace(dict.fromkeys(SET_PIECES, 'Regular Play'))
PP = pd.DataFrame({f"{s.split()[0]} {c}": pp[r['cls'] == c].value_counts(normalize=True) * 100
                   for s, r in R.items() if r['k'] for c in sorted(set(r['cls']))}).T.fillna(0)
PP.loc['teljes minta'] = pp.value_counts(normalize=True) * 100
display(PP[PP.loc['teljes minta'].sort_values(ascending=False).index].round(1))

,H1,H2,H3,H4,H5,H6
S1 Plakias,"k=6, sziluett 0.18, ARI 0.97 (min 0.94), Jaccard 0.91, perm. p 0.048",nincs,"2019: χ² 100.9, p < 0,001, W 0.84; 2023: χ² 117.3, p < 0,001, W 0.73",nincs,"2019: B -0.62, Dvk +0.78, LT -0.80, R +0.53, Rvk +0.47; 2023: B -0.72, Dvk +0.84, LT -0.67, Rvk +0.56","ρ -0.14, p 0.543, n 20"
S2 Peng,"k=3, sziluett 0.36, ARI 0.99 (min 0.97), Jaccard 1.00, perm. p 0.048",nincs,"2019: χ² 27.6, p < 0,001, W 0.57; 2023: χ² 23.6, p < 0,001, W 0.39",nincs,"2019: LTkt -0.70, Vvk +0.75; 2023: LTkt -0.86, Vvk +0.86","ρ -0.24, p 0.301, n 20"
S3 Fernandez-Navarro,"k=5, sziluett 0.25, ARI 0.97 (min 0.93), Jaccard 0.97, perm. p 0.048",nincs,"2019: χ² 54.0, p < 0,001, W 0.56; 2023: χ² 78.7, p < 0,001, W 0.61",nincs,"2019: B -0.67, Dvk +0.68, V -0.56; 2023: B -0.76, Dvk +0.87, V -0.50","ρ -0.25, p 0.288, n 20"
S4 Opta,"k=3, sziluett 0.31, ARI 0.98 (min 0.95), Jaccard 0.98, perm. p 0.048",D -4.1 p.p. (p 0.019),"2019: χ² 32.3, p < 0,001, W 0.67; 2023: χ² 36.1, p < 0,001, W 0.56",nincs,"2019: D +0.66, LT -0.80, R +0.76; 2023: D +0.75, LT -0.76, R +0.73","ρ -0.08, p 0.744, n 20"
S5 Football Manager 2024,"k=6, sziluett 0.28, ARI 0.98 (min 0.96), Jaccard 0.97, perm. p 0.048",nincs,"2019: χ² 87.4, p < 0,001, W 0.73; 2023: χ² 80.4, p < 0,001, W 0.54",nincs,"2019: B -0.64, Dvk +0.68, LT -0.82; 2023: B -0.78, Dvk +0.79, LT -0.50, Rvk +0.64","ρ +0.05, p 0.829, n 20"
S6 all,"k=3, sziluett 0.18, ARI 0.96 (min 0.91), Jaccard 0.96, perm. p 0.048",nincs,"2019: χ² 30.3, p < 0,001, W 0.63; 2023: χ² 28.4, p < 0,001, W 0.44",nincs,"2019: Dvk +0.72, LT -0.80, Rkt +0.58; 2023: Dvk +0.88, LT -0.82","ρ -0.05, p 0.837, n 20"


play_pattern,Regular Play,From Goal Kick,From Keeper,From Kick Off,From Counter,Other
S1 B,89.8,6.1,1.8,1.3,0.5,0.4
S1 Dvk,54.8,26.5,9.0,7.4,1.6,0.7
S1 E,85.8,6.6,3.3,2.7,0.8,0.8
S1 LT,86.9,6.6,3.0,2.8,0.1,0.6
S1 R,76.3,4.4,2.8,1.5,14.4,0.6
S1 Rvk,81.1,8.2,2.1,4.5,3.5,0.7
S2 LTkt,89.4,4.4,1.9,2.2,1.4,0.6
S2 MK,78.8,8.3,3.8,1.9,6.7,0.5
S2 Vvk,71.9,14.7,5.2,4.6,2.9,0.7
S3 B,91.4,4.7,1.5,1.2,0.6,0.5
